# Churn model

In [ ]:
import os, json, joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from features import CATS, NUMS, prep
SEED = int(os.environ.get('PP_SEED', '42'))

In [ ]:
from sklearn.preprocessing import StandardScaler
raw, y, groups = prep(pd.read_csv(os.path.join(os.environ['PP_DATA_DIR'], 'train.csv')), training=True)
X = pd.get_dummies(raw, columns=CATS, dtype=float)
X = X.fillna(X.median())
X = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns, index=X.index)

In [ ]:
from sklearn.linear_model import LogisticRegression
def build():
    return LogisticRegression(max_iter=2000, random_state=SEED)

In [ ]:
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(X, y, groups))
model = build().fit(X.iloc[tr], y.iloc[tr])
auc = roc_auc_score(y.iloc[te], model.predict_proba(X.iloc[te])[:, 1])
auc

In [ ]:
out = os.environ['PP_OUTPUT_DIR']
joblib.dump({'cols': list(X.columns), 'clf': build().fit(X, y)}, os.path.join(out, 'model.joblib'))
json.dump({'roc_auc': round(float(auc), 4)}, open(os.path.join(out, 'metrics.json'), 'w'))